# Strategic Compensation & Pay Equity Analytics

## 03 — Data Cleaning

### Objective

Prepare accurate, consistent, and analysis-ready employee, compensation, salary band, performance, and promotion datasets based on findings from the Data Quality Audit.

### Cleaning Approach

- Preserve the original source CSV files.
- Perform cleaning using Python and Pandas.
- Resolve confirmed duplicates and missing values using documented rules.
- Investigate historical date inconsistencies before applying corrections.
- Retain legitimate compensation differences across job levels.
- Validate all cleaned datasets before loading them into SQL Server.

**Data Storage Strategy:** All cleaning will be performed in memory. No processed CSV files will be exported.

In [1]:

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)


Pandas Version: 3.0.6
NumPy Version: 2.5.4


In [2]:

# Project and data paths
project_path = Path.cwd().parent
data_path = project_path / "data"

# Load raw datasets
employees = pd.read_csv(data_path / "employees.csv")
jobs = pd.read_csv(data_path / "jobs.csv")
compensation = pd.read_csv(data_path / "compensation.csv")
salary_bands = pd.read_csv(data_path / "salary_bands.csv")
performance = pd.read_csv(data_path / "performance.csv")
promotions = pd.read_csv(data_path / "promotions.csv")

# Store datasets in a dictionary
datasets = {
    "Employees": employees,
    "Jobs": jobs,
    "Compensation": compensation,
    "Salary Bands": salary_bands,
    "Performance": performance,
    "Promotions": promotions
}

# Verify loaded dimensions
dataset_summary = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(dataset_summary)


,Dataset,Rows,Columns
0,Employees,10000,11
1,Jobs,50,7
2,Compensation,30018,7
3,Salary Bands,300,7
4,Performance,30000,4
5,Promotions,2737,5


In [3]:

# Count records before cleaning
rows_before = len(compensation)

# Count exact duplicate records
duplicates_before = compensation.duplicated().sum()

# Remove exact duplicates
compensation = compensation.drop_duplicates().reset_index(drop=True)

# Count records after cleaning
rows_after = len(compensation)

# Verify cleaning
duplicate_summary = pd.DataFrame({
    "Check": [
        "Records Before Cleaning",
        "Exact Duplicates Identified",
        "Records After Cleaning",
        "Duplicates Remaining",
        "Records Removed"
    ],
    "Result": [
        rows_before,
        duplicates_before,
        rows_after,
        compensation.duplicated().sum(),
        rows_before - rows_after
    ]
})

display(duplicate_summary)


,Check,Result
0,Records Before Cleaning,30018
1,Exact Duplicates Identified,18
2,Records After Cleaning,30000
3,Duplicates Remaining,0
4,Records Removed,18


### Step 2 — Compensation Duplicate Removal

**Objective:** Remove exact duplicate compensation records to prevent inflated financial metrics.

**Cleaning Method:**
- Identified duplicate records using `duplicated()`.
- Removed exact duplicates using `drop_duplicates()`.
- Retained the first occurrence of each record.

**Results:**
- Original compensation records: 30,018.
- Duplicate records removed: 18.
- Final compensation records: 30,000.
- Remaining exact duplicates: 0.

**Business Impact:**

Removing duplicate records prevents double-counting of employee compensation and improves the reliability of salary benchmarking, payroll analysis, and pay equity calculations.

**Cleaning Decision:** Successfully removed all confirmed exact duplicates without modifying the original source CSV.

In [4]:

# Examine employees with missing education
missing_education = employees[
    employees["Education_Level"].isna()
]

# Examine employees with missing experience
missing_experience = employees[
    employees["Years_Experience"].isna()
]

print("Missing Education Level:", len(missing_education))
print("Missing Years of Experience:", len(missing_experience))

print("\nEmployees With Missing Experience:")

display(
    missing_experience[
        [
            "Employee_ID",
            "Age",
            "Hire_Date",
            "Tenure_Years_2026",
            "Years_Experience"
        ]
    ].head(10)
)

print("\nEducation Level Distribution:")

display(
    employees["Education_Level"]
    .value_counts(dropna=False)
)


Missing Education Level: 45
Missing Years of Experience: 25

Employees With Missing Experience:


,Employee_ID,Age,Hire_Date,Tenure_Years_2026,Years_Experience
788,E00789,30,2022-11-16,4.1,NaN
833,E00834,29,2023-03-15,3.8,NaN
1321,E01322,37,2023-10-11,3.2,NaN
1704,E01705,37,2021-09-23,5.3,NaN
1830,E01831,36,2026-01-21,0.9,NaN
2478,E02479,43,2025-01-07,2.0,NaN
2483,E02484,27,2023-10-20,3.2,NaN
2969,E02970,36,2018-04-24,8.7,NaN
3296,E03297,35,2020-01-25,6.9,NaN
3405,E03406,34,2017-12-08,9.1,NaN



Education Level Distribution:


Education_Level
Bachelor's    5162
Master's      2450
MBA           1469
Diploma        679
Doctorate      195
NaN             45
Name: count, dtype: int64

In [5]:

# Fill missing education values
employees["Education_Level"] = (
    employees["Education_Level"].fillna("Unknown")
)

# Keep missing experience as NaN
# We will not estimate total experience from tenure

# Validate missing values
employee_cleaning_summary = pd.DataFrame({
    "Column": [
        "Education_Level",
        "Years_Experience"
    ],
    "Missing_After_Cleaning": [
        employees["Education_Level"].isna().sum(),
        employees["Years_Experience"].isna().sum()
    ],
    "Cleaning_Decision": [
        "Filled with Unknown",
        "Retained as NaN"
    ]
})

display(employee_cleaning_summary)

print(
    "\nUnknown Education Records:",
    (employees["Education_Level"] == "Unknown").sum()
)

print("Total Employees:", len(employees))


,Column,Missing_After_Cleaning,Cleaning_Decision
0,Education_Level,0,Filled with Unknown
1,Years_Experience,25,Retained as NaN



Unknown Education Records: 45
Total Employees: 10000


### Step 3 — Employee Missing-Value Treatment

**Objective:** Handle missing employee information while preserving data accuracy and employee coverage.

**Cleaning Actions:**
- Replaced 45 missing Education_Level values with "Unknown".
- Retained 25 missing Years_Experience values as NaN.
- Preserved all 10,000 employee records.

**Business Justification:**

Missing education levels were assigned an explicit "Unknown" category rather than an assumed qualification.

Missing professional experience values were retained because organizational tenure does not necessarily represent total professional experience.

**Validation Results:**
- Missing Education_Level values remaining: 0.
- Missing Years_Experience values remaining: 25.
- Total employee records: 10,000.

**Cleaning Decision:**

Employee information is ready for subsequent cleaning and analysis. Records with missing professional experience will be excluded only from calculations requiring that field, with the reduced sample size documented.

In [6]:

# Select compensation records with missing annual bonuses
missing_bonus = compensation[
    compensation["Annual_Bonus"].isna()
].copy()

# Calculate the implied annual bonus
missing_bonus["Calculated_Bonus"] = (
    missing_bonus["Total_Compensation"]
    - missing_bonus["Base_Salary"]
    - missing_bonus["Allowances"]
).round(2)

# Check whether the calculated bonus is valid
missing_bonus["Valid_Calculated_Bonus"] = (
    missing_bonus["Calculated_Bonus"] >= 0
)

# Summary
bonus_investigation = pd.DataFrame({
    "Check": [
        "Missing Annual Bonus Records",
        "Recoverable Non-Negative Bonuses",
        "Negative Calculated Bonuses",
        "Missing Calculated Bonus Values"
    ],
    "Result": [
        len(missing_bonus),
        int(missing_bonus["Valid_Calculated_Bonus"].sum()),
        int((missing_bonus["Calculated_Bonus"] < 0).sum()),
        int(missing_bonus["Calculated_Bonus"].isna().sum())
    ]
})

display(bonus_investigation)

print("\nSample Missing Bonus Records:")

display(
    missing_bonus[
        [
            "Employee_ID",
            "Compensation_Year",
            "Base_Salary",
            "Allowances",
            "Total_Compensation",
            "Calculated_Bonus"
        ]
    ].head(10)
)


,Check,Result
0,Missing Annual Bonus Records,75
1,Recoverable Non-Negative Bonuses,75
2,Negative Calculated Bonuses,0
3,Missing Calculated Bonus Values,0



Sample Missing Bonus Records:


,Employee_ID,Compensation_Year,Base_Salary,Allowances,Total_Compensation,Calculated_Bonus
115,E00116,2024,1892837.20,74516.25,2216316.97,248963.52
126,E00127,2024,1753415.43,89166.21,2072396.48,229814.84
1248,E01249,2024,383075.15,21005.64,423424.82,19344.03
1865,E01866,2024,615403.98,31490.90,683901.86,37006.98
3400,E03401,2024,5261514.57,228864.49,6374875.12,884496.06
3717,E03718,2024,734894.99,26162.35,820230.38,59173.04
3968,E03969,2024,677507.17,13218.63,731686.44,40960.64
4648,E04649,2024,3365435.26,56605.15,3775455.39,353414.98
5040,E05041,2024,1526563.20,33818.45,1661443.27,101061.62
5288,E05289,2024,5072004.06,235950.76,6095982.96,788028.14


In [7]:

# Count missing bonuses before cleaning
missing_before = compensation["Annual_Bonus"].isna().sum()

# Identify missing bonus records
missing_mask = compensation["Annual_Bonus"].isna()

# Recover missing bonuses using compensation components
compensation.loc[missing_mask, "Annual_Bonus"] = (
    compensation.loc[missing_mask, "Total_Compensation"]
    - compensation.loc[missing_mask, "Base_Salary"]
    - compensation.loc[missing_mask, "Allowances"]
).round(2)

# Validate recovered values
missing_after = compensation["Annual_Bonus"].isna().sum()

compensation["Calculated_Total"] = (
    compensation["Base_Salary"]
    + compensation["Annual_Bonus"]
    + compensation["Allowances"]
).round(2)

compensation["Reconciliation_Difference"] = (
    compensation["Total_Compensation"]
    - compensation["Calculated_Total"]
).round(2)

bonus_cleaning_summary = pd.DataFrame({
    "Check": [
        "Missing Bonuses Before Cleaning",
        "Bonuses Recovered",
        "Missing Bonuses After Cleaning",
        "Negative Annual Bonuses",
        "Reconciliation Differences Above ₹0.01",
        "Total Compensation Records"
    ],
    "Result": [
        missing_before,
        missing_before - missing_after,
        missing_after,
        int((compensation["Annual_Bonus"] < 0).sum()),
        int(
            (
                compensation["Reconciliation_Difference"].abs()
                > 0.010001
            ).sum()
        ),
        len(compensation)
    ]
})

display(bonus_cleaning_summary)


,Check,Result
0,Missing Bonuses Before Cleaning,75
1,Bonuses Recovered,75
2,Missing Bonuses After Cleaning,0
3,Negative Annual Bonuses,0
4,Reconciliation Differences Above ₹0.01,0
5,Total Compensation Records,30000


### Step 4 — Missing Annual Bonus Treatment

**Objective:** Recover missing annual bonus values using existing compensation information.

**Cleaning Method:**

Annual Bonus = Total Compensation − Base Salary − Allowances

**Results:**
- Missing annual bonuses identified: 75.
- Missing annual bonuses recovered: 75.
- Missing annual bonuses remaining: 0.
- Negative annual bonuses: 0.
- Compensation reconciliation differences above ₹0.01: 0.
- Total compensation records retained: 30,000.

**Business Interpretation:**

All missing annual bonus values were reconstructed from existing financial components rather than replaced with arbitrary averages or zeros.

This preserves compensation consistency and improves the reliability of salary benchmarking and pay equity calculations.

**Cleaning Decision:**

Retain the recovered bonus values. No compensation records were removed during this step.

In [8]:

# Count missing ratings before cleaning
missing_before = performance["Performance_Rating"].isna().sum()

# Define validated category-to-rating mapping
rating_mapping = {
    "Needs Improvement": 1,
    "Developing": 2,
    "Meets Expectations": 3,
    "Exceeds Expectations": 4,
    "Outstanding": 5
}

# Fill missing ratings using performance categories
performance["Performance_Rating"] = (
    performance["Performance_Rating"].fillna(
        performance["Performance_Category"].map(rating_mapping)
    )
)

# Validate results
missing_after = performance["Performance_Rating"].isna().sum()

rating_mismatches = (
    performance["Performance_Rating"]
    != performance["Performance_Category"].map(rating_mapping)
).sum()

performance_cleaning_summary = pd.DataFrame({
    "Check": [
        "Missing Ratings Before Cleaning",
        "Ratings Recovered",
        "Missing Ratings After Cleaning",
        "Rating-Category Mismatches",
        "Total Performance Records"
    ],
    "Result": [
        missing_before,
        missing_before - missing_after,
        missing_after,
        int(rating_mismatches),
        len(performance)
    ]
})

display(performance_cleaning_summary)


,Check,Result
0,Missing Ratings Before Cleaning,60
1,Ratings Recovered,60
2,Missing Ratings After Cleaning,0
3,Rating-Category Mismatches,0
4,Total Performance Records,30000


### Step 5 — Missing Performance Rating Treatment

**Objective:** Recover missing employee performance ratings using validated performance categories.

**Cleaning Method:**
- Established a category-to-rating mapping using the Data Quality Audit findings.
- Recovered missing ratings from their corresponding performance categories.
- Validated that every numeric rating matches its category.

**Results:**
- Missing performance ratings identified: 60.
- Ratings recovered: 60.
- Missing ratings remaining: 0.
- Rating-category mismatches: 0.
- Total performance records retained: 30,000.

**Business Interpretation:**

The missing performance ratings were recovered using a consistent relationship already present in the dataset, avoiding arbitrary statistical imputation.

**Cleaning Decision:**

Retain all recovered ratings. No performance records were removed during this step.

In [9]:

# Prepare employee hire years
hire_years = employees[[
    "Employee_ID",
    "Hire_Date"
]].copy()

hire_years["Hire_Year"] = pd.to_datetime(
    hire_years["Hire_Date"],
    errors="coerce"
).dt.year

# Check compensation records
comp_hire_check = compensation.merge(
    hire_years[["Employee_ID", "Hire_Year"]],
    on="Employee_ID",
    how="left"
)

invalid_comp = comp_hire_check[
    comp_hire_check["Compensation_Year"]
    < comp_hire_check["Hire_Year"]
]

# Check performance records
perf_hire_check = performance.merge(
    hire_years[["Employee_ID", "Hire_Year"]],
    on="Employee_ID",
    how="left"
)

invalid_perf = perf_hire_check[
    perf_hire_check["Review_Year"]
    < perf_hire_check["Hire_Year"]
]

# Compare affected employee-year combinations
invalid_comp_keys = set(
    zip(invalid_comp["Employee_ID"],
        invalid_comp["Compensation_Year"])
)

invalid_perf_keys = set(
    zip(invalid_perf["Employee_ID"],
        invalid_perf["Review_Year"])
)

historical_summary = pd.DataFrame({
    "Check": [
        "Invalid Compensation Records",
        "Invalid Performance Records",
        "Shared Invalid Employee-Year Keys",
        "Compensation-Only Invalid Keys",
        "Performance-Only Invalid Keys"
    ],
    "Result": [
        len(invalid_comp),
        len(invalid_perf),
        len(invalid_comp_keys & invalid_perf_keys),
        len(invalid_comp_keys - invalid_perf_keys),
        len(invalid_perf_keys - invalid_comp_keys)
    ]
})

display(historical_summary)


,Check,Result
0,Invalid Compensation Records,3695
1,Invalid Performance Records,3695
2,Shared Invalid Employee-Year Keys,3695
3,Compensation-Only Invalid Keys,0
4,Performance-Only Invalid Keys,0


In [10]:

# Invalid compensation records by year
invalid_by_year = (
    invalid_comp
    .groupby("Compensation_Year")
    .agg(
        Invalid_Records=("Employee_ID", "size"),
        Affected_Employees=("Employee_ID", "nunique")
    )
    .reset_index()
)

print("Invalid Compensation Records by Year:")
display(invalid_by_year)

# Number of distinct employees affected
affected_employees = invalid_comp["Employee_ID"].nunique()

# Count invalid years per employee
invalid_years_per_employee = (
    invalid_comp
    .groupby("Employee_ID")["Compensation_Year"]
    .nunique()
    .value_counts()
    .sort_index()
    .reset_index()
)

invalid_years_per_employee.columns = [
    "Invalid_Years",
    "Employee_Count"
]

print("\nEmployees Affected by Invalid Historical Records:")
print("Unique Affected Employees:", affected_employees)

display(invalid_years_per_employee)


Invalid Compensation Records by Year:


,Compensation_Year,Invalid_Records,Affected_Employees
0,2024,2461,2461
1,2025,1234,1234



Employees Affected by Invalid Historical Records:
Unique Affected Employees: 2461


,Invalid_Years,Employee_Count
0,1,1227
1,2,1234


In [11]:

# Record counts before cleaning
comp_before = len(compensation)
perf_before = len(performance)

# Keep compensation records from hire year onward
valid_comp_ids = comp_hire_check.loc[
    comp_hire_check["Compensation_Year"]
    >= comp_hire_check["Hire_Year"],
    ["Employee_ID", "Compensation_Year"]
]

compensation = compensation.merge(
    valid_comp_ids,
    on=["Employee_ID", "Compensation_Year"],
    how="inner"
)

# Keep performance records from hire year onward
valid_perf_ids = perf_hire_check.loc[
    perf_hire_check["Review_Year"]
    >= perf_hire_check["Hire_Year"],
    ["Employee_ID", "Review_Year"]
]

performance = performance.merge(
    valid_perf_ids,
    on=["Employee_ID", "Review_Year"],
    how="inner"
)

# Cleaning summary
historical_cleaning_summary = pd.DataFrame({
    "Dataset": ["Compensation", "Performance"],
    "Records_Before": [comp_before, perf_before],
    "Records_After": [
        len(compensation),
        len(performance)
    ],
    "Records_Removed": [
        comp_before - len(compensation),
        perf_before - len(performance)
    ]
})

display(historical_cleaning_summary)


,Dataset,Records_Before,Records_After,Records_Removed
0,Compensation,30000,26305,3695
1,Performance,30000,26305,3695


In [12]:

# Count promotions before cleaning
promotions_before = len(promotions)

# Prepare employee hire dates
employee_hire_dates = employees[
    ["Employee_ID", "Hire_Date"]
].copy()

employee_hire_dates["Hire_Date"] = pd.to_datetime(
    employee_hire_dates["Hire_Date"]
)

# Match promotions with employee hire dates
promotion_date_check = promotions.merge(
    employee_hire_dates,
    on="Employee_ID",
    how="left"
)

# Convert promotion dates
promotion_date_check["Promotion_Date"] = pd.to_datetime(
    promotion_date_check["Promotion_Date"]
)

# Keep promotions occurring on or after hire date
valid_promotions = promotion_date_check[
    promotion_date_check["Promotion_Date"]
    >= promotion_date_check["Hire_Date"]
].copy()

# Preserve original promotion columns
promotions = valid_promotions[
    promotions.columns
].reset_index(drop=True)

# Validate cleaning results
promotion_cleaning_summary = pd.DataFrame({
    "Check": [
        "Promotions Before Cleaning",
        "Invalid Promotions Removed",
        "Promotions After Cleaning",
        "Remaining Promotions Before Hire Date"
    ],
    "Result": [
        promotions_before,
        promotions_before - len(promotions),
        len(promotions),
        int(
            (
                valid_promotions["Promotion_Date"]
                < valid_promotions["Hire_Date"]
            ).sum()
        )
    ]
})

display(promotion_cleaning_summary)


,Check,Result
0,Promotions Before Cleaning,2737
1,Invalid Promotions Removed,132
2,Promotions After Cleaning,2605
3,Remaining Promotions Before Hire Date,0


### Step 6 — Historical Record Cleaning

**Objective:** Remove records that violate employee employment-date eligibility rules.

**Cleaning Rules:**
- Compensation_Year ≥ Hire_Year.
- Review_Year ≥ Hire_Year.
- Promotion_Date ≥ Hire_Date.

**Cleaning Results:**

| Dataset | Records Removed | Records Retained |
|---|---:|---:|
| Compensation | 3,695 | 26,305 |
| Performance | 3,695 | 26,305 |
| Promotions | 132 | 2,605 |

**Business Interpretation:**

Removing pre-hire records improves the reliability of compensation trends, performance analysis, and employee promotion metrics.

Compensation and performance records were filtered using the same employment-year eligibility rule, preserving consistency between the two datasets.

**Analytical Consideration:**

Employee coverage varies across reporting years because employees hired later are not eligible for earlier-year records.

**Cleaning Decision:**

Exclude pre-hire records from the analytical datasets while preserving the original source CSV files.

In [13]:

# Select employees with available experience values
experience_check = employees[
    employees["Years_Experience"].notna()
].copy()

# Calculate difference between tenure and experience
experience_check["Experience_Gap"] = (
    experience_check["Tenure_Years_2026"]
    - experience_check["Years_Experience"]
).round(2)

# Identify employees whose tenure exceeds experience
experience_check["Tenure_Exceeds_Experience"] = (
    experience_check["Experience_Gap"] > 0
)

# Summarize the inconsistencies
experience_summary = pd.DataFrame({
    "Check": [
        "Employees With Available Experience",
        "Tenure Exceeds Experience",
        "Gap Above 1 Year",
        "Gap Above 2 Years",
        "Maximum Experience Gap",
        "Average Positive Experience Gap"
    ],
    "Result": [
        len(experience_check),
        int(experience_check["Tenure_Exceeds_Experience"].sum()),
        int((experience_check["Experience_Gap"] > 1).sum()),
        int((experience_check["Experience_Gap"] > 2).sum()),
        round(experience_check["Experience_Gap"].max(), 2),
        round(
            experience_check.loc[
                experience_check["Experience_Gap"] > 0,
                "Experience_Gap"
            ].mean(), 2
        )
    ]
})

display(experience_summary)


,Check,Result
0,Employees With Available Experience,9975.00
1,Tenure Exceeds Experience,2332.00
2,Gap Above 1 Year,1078.00
3,Gap Above 2 Years,0.00
4,Maximum Experience Gap,2.00
5,Average Positive Experience Gap,1.01


In [14]:

# Check whether experience is recorded in whole years
experience_check["Experience_Is_Whole_Year"] = (
    experience_check["Years_Experience"] % 1 == 0
)

# Compare experience and tenure precision
precision_summary = pd.DataFrame({
    "Check": [
        "Available Experience Records",
        "Whole-Year Experience Values",
        "Decimal Experience Values",
        "Tenure Exceeds Experience by 0–1 Year",
        "Tenure Exceeds Experience by More Than 1 Year"
    ],
    "Result": [
        len(experience_check),
        int(experience_check["Experience_Is_Whole_Year"].sum()),
        int((~experience_check["Experience_Is_Whole_Year"]).sum()),
        int(
            (
                (experience_check["Experience_Gap"] > 0)
                & (experience_check["Experience_Gap"] <= 1)
            ).sum()
        ),
        int((experience_check["Experience_Gap"] > 1).sum())
    ]
})

display(precision_summary)

print("\nSample Experience and Tenure Differences:")

display(
    experience_check.loc[
        experience_check["Tenure_Exceeds_Experience"],
        [
            "Employee_ID",
            "Age",
            "Years_Experience",
            "Tenure_Years_2026",
            "Experience_Gap"
        ]
    ].head(10)
)


,Check,Result
0,Available Experience Records,9975
1,Whole-Year Experience Values,9975
2,Decimal Experience Values,0
3,Tenure Exceeds Experience by 0–1 Year,1254
4,Tenure Exceeds Experience by More Than 1 Year,1078



Sample Experience and Tenure Differences:


,Employee_ID,Age,Years_Experience,Tenure_Years_2026,Experience_Gap
0,E00001,21,0.0,1.9,1.9
2,E00003,27,2.0,2.6,0.6
6,E00007,27,3.0,3.7,0.7
11,E00012,22,1.0,1.7,0.7
14,E00015,22,1.0,2.3,1.3
16,E00017,36,13.0,14.6,1.6
17,E00018,32,8.0,8.8,0.8
22,E00023,23,0.0,1.2,1.2
26,E00027,21,0.0,0.3,0.3
29,E00030,24,0.0,0.7,0.7


In [15]:

# Create an experience quality flag
employees["Experience_Quality_Flag"] = "Valid"

# Flag missing experience
employees.loc[
    employees["Years_Experience"].isna(),
    "Experience_Quality_Flag"
] = "Missing"

# Flag tenure exceeding reported experience
employees.loc[
    employees["Years_Experience"].notna()
    & (
        employees["Tenure_Years_2026"]
        > employees["Years_Experience"]
    ),
    "Experience_Quality_Flag"
] = "Review"

# Summarize quality flags
experience_flag_summary = (
    employees["Experience_Quality_Flag"]
    .value_counts()
    .rename_axis("Quality_Flag")
    .reset_index(name="Employee_Count")
)

display(experience_flag_summary)

print("Total Employees:", len(employees))
print(
    "Missing Experience:",
    employees["Years_Experience"].isna().sum()
)


,Quality_Flag,Employee_Count
0,Valid,7643
1,Review,2332
2,Missing,25


Total Employees: 10000
Missing Experience: 25


### Step 7 — Experience and Tenure Consistency Treatment

**Objective:** Address inconsistencies between reported professional experience and organizational tenure without introducing unsupported corrections.

**Investigation Findings:**
- Employees with available professional experience: 9,975.
- Employees whose tenure exceeds reported experience: 2,332.
- Employees with missing experience: 25.
- All available professional experience values are recorded in whole years.
- Maximum tenure-versus-experience difference: 2 years.

**Cleaning Approach:**

Created an `Experience_Quality_Flag` column with three classifications:

- **Valid:** 7,643 employees.
- **Review:** 2,332 employees.
- **Missing:** 25 employees.

**Business Interpretation:**

Employee tenure was previously validated against hire dates. However, reported professional experience contains inconsistencies that cannot all be explained by differences in numerical precision.

**Cleaning Decision:**

Retain original professional experience values and flag potentially inconsistent records instead of applying unsupported corrections.

For experience-adjusted compensation analysis, use validated experience records and clearly document any excluded observations.

In [16]:

# Recalculate compensation totals
compensation["Calculated_Total"] = (
    compensation["Base_Salary"]
    + compensation["Annual_Bonus"]
    + compensation["Allowances"]
).round(2)

# Calculate reconciliation differences
compensation["Reconciliation_Difference"] = (
    compensation["Total_Compensation"]
    - compensation["Calculated_Total"]
).round(2)

# Define rounding tolerance
tolerance = 0.010001

# Validate compensation records
compensation_validation = pd.DataFrame({
    "Check": [
        "Total Compensation Records",
        "Missing Base Salary",
        "Missing Annual Bonus",
        "Missing Allowances",
        "Missing Total Compensation",
        "Negative Compensation Components",
        "Reconciliation Differences Above ₹0.01",
        "Maximum Absolute Reconciliation Difference"
    ],
    "Result": [
        len(compensation),
        int(compensation["Base_Salary"].isna().sum()),
        int(compensation["Annual_Bonus"].isna().sum()),
        int(compensation["Allowances"].isna().sum()),
        int(compensation["Total_Compensation"].isna().sum()),
        int(
            (
                compensation[
                    ["Base_Salary", "Annual_Bonus", "Allowances"]
                ] < 0
            ).sum().sum()
        ),
        int(
            (
                compensation["Reconciliation_Difference"].abs()
                > tolerance
            ).sum()
        ),
        compensation["Reconciliation_Difference"].abs().max()
    ]
})

display(compensation_validation)


,Check,Result
0,Total Compensation Records,26305.00
1,Missing Base Salary,0.00
2,Missing Annual Bonus,0.00
3,Missing Allowances,0.00
4,Missing Total Compensation,0.00
5,Negative Compensation Components,0.00
6,Reconciliation Differences Above ₹0.01,0.00
7,Maximum Absolute Reconciliation Difference,0.01


### Step 8 — Compensation Reconciliation and Validation

**Objective:** Validate financial consistency after compensation data cleaning.

**Formula:**

Total Compensation = Base Salary + Annual Bonus + Allowances

**Validation Results:**
- Total compensation records: 26,305.
- Missing compensation values: 0.
- Negative compensation components: 0.
- Reconciliation differences exceeding ₹0.01: 0.
- Maximum absolute reconciliation difference: ₹0.01.

**Business Interpretation:**

All cleaned compensation records satisfy the financial reconciliation rule within the accepted rounding tolerance.

The remaining differences are consistent with minor monetary rounding effects.

**Cleaning Decision:**

Retain the original compensation amounts. No additional financial adjustments are required.

In [17]:

# Update dictionary with cleaned DataFrames
datasets = {
    "Employees": employees,
    "Jobs": jobs,
    "Compensation": compensation,
    "Salary Bands": salary_bands,
    "Performance": performance,
    "Promotions": promotions
}

# Define expected unique keys
dataset_keys = {
    "Employees": ["Employee_ID"],
    "Jobs": ["Job_ID"],
    "Compensation": ["Employee_ID", "Compensation_Year"],
    "Salary Bands": ["Job_ID", "Location"],
    "Performance": ["Employee_ID", "Review_Year"],
    "Promotions": ["Employee_ID", "Promotion_Date"]
}

# Validate cleaned datasets
integrity_results = []

for name, df in datasets.items():

    keys = dataset_keys[name]

    integrity_results.append({
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1],
        "Duplicate_Keys": int(df.duplicated(subset=keys).sum()),
        "Missing_Key_Values": int(df[keys].isna().sum().sum()),
        "Missing_Cells": int(df.isna().sum().sum())
    })

integrity_summary = pd.DataFrame(integrity_results)

display(integrity_summary)


,Dataset,Rows,Columns,Duplicate_Keys,Missing_Key_Values,Missing_Cells
0,Employees,10000,12,0,0,25
1,Jobs,50,7,0,0,0
2,Compensation,26305,9,0,0,0
3,Salary Bands,300,7,0,0,0
4,Performance,26305,4,0,0,0
5,Promotions,2605,5,0,0,0


### Step 9 — Final Dataset Integrity Validation

**Objective:** Verify the structural integrity of all six datasets following data cleaning.

**Validation Results:**
- All six datasets have the expected record counts.
- Duplicate primary and composite keys: 0.
- Missing key values: 0.
- Remaining missing values: 25, all in Years_Experience.
- Compensation and Performance each contain 26,305 eligible annual records.
- Promotions contains 2,605 valid-date records.

**Business Interpretation:**

The cleaned datasets demonstrate strong structural integrity. The remaining missing experience values are intentionally retained to avoid unsupported assumptions.

**Cleaning Decision:**

Preserve Experience_Quality_Flag for analytical transparency and remove temporary compensation reconciliation columns before finalizing the dataset schema.

In [18]:

# Remove temporary compensation validation columns
compensation = compensation.drop(
    columns=[
        "Calculated_Total",
        "Reconciliation_Difference"
    ],
    errors="ignore"
)

# Update cleaned dataset dictionary
datasets["Compensation"] = compensation

# Verify final structures
final_structure = pd.DataFrame([
    {
        "Dataset": name,
        "Rows": len(df),
        "Columns": df.shape[1]
    }
    for name, df in datasets.items()
])

display(final_structure)

print(
    "\nTemporary Compensation Columns Remaining:",
    [
        col for col in [
            "Calculated_Total",
            "Reconciliation_Difference"
        ]
        if col in compensation.columns
    ]
)


,Dataset,Rows,Columns
0,Employees,10000,12
1,Jobs,50,7
2,Compensation,26305,7
3,Salary Bands,300,7
4,Performance,26305,4
5,Promotions,2605,5



Temporary Compensation Columns Remaining: []


In [19]:

# Original dataset record counts
original_counts = {
    "Employees": 10000,
    "Jobs": 50,
    "Compensation": 30018,
    "Salary Bands": 300,
    "Performance": 30000,
    "Promotions": 2737
}

# Prepare final cleaning summary
final_cleaning_summary = []

for name, df in datasets.items():

    original_rows = original_counts[name]
    cleaned_rows = len(df)

    final_cleaning_summary.append({
        "Dataset": name,
        "Original_Rows": original_rows,
        "Cleaned_Rows": cleaned_rows,
        "Rows_Removed": original_rows - cleaned_rows,
        "Duplicate_Keys": int(
            df.duplicated(subset=dataset_keys[name]).sum()
        ),
        "Missing_Key_Values": int(
            df[dataset_keys[name]].isna().sum().sum()
        ),
        "Remaining_Missing_Cells": int(
            df.isna().sum().sum()
        )
    })

final_cleaning_summary = pd.DataFrame(
    final_cleaning_summary
)

display(final_cleaning_summary)


,Dataset,Original_Rows,Cleaned_Rows,Rows_Removed,Duplicate_Keys,Missing_Key_Values,Remaining_Missing_Cells
0,Employees,10000,10000,0,0,0,25
1,Jobs,50,50,0,0,0,0
2,Compensation,30018,26305,3713,0,0,0
3,Salary Bands,300,300,0,0,0,0
4,Performance,30000,26305,3695,0,0,0
5,Promotions,2737,2605,132,0,0,0


In [20]:

# Inspect final cleaned dataset schemas

for name, df in datasets.items():
    print(f"\n{'=' * 45}")
    print(f"{name.upper()} — {len(df):,} rows")
    print(f"{'=' * 45}")

    schema = pd.DataFrame({
        "Column": df.columns,
        "Data_Type": df.dtypes.astype(str).values,
        "Missing_Values": df.isna().sum().values
    })

    display(schema)



EMPLOYEES — 10,000 rows


,Column,Data_Type,Missing_Values
0,Employee_ID,str,0
1,Gender,str,0
2,Age,int64,0
3,Education_Level,str,0
4,Location,str,0
5,Hire_Date,str,0
6,Years_Experience,float64,25
7,Tenure_Years_2026,float64,0
8,Job_ID,str,0
9,Employment_Type,str,0



JOBS — 50 rows


,Column,Data_Type,Missing_Values
0,Job_ID,str,0
1,Job_Title,str,0
2,Department,str,0
3,Job_Family,str,0
4,Job_Level,int64,0
5,Min_Experience,int64,0
6,Max_Experience,int64,0



COMPENSATION — 26,305 rows


,Column,Data_Type,Missing_Values
0,Employee_ID,str,0
1,Compensation_Year,int64,0
2,Base_Salary,float64,0
3,Annual_Bonus,float64,0
4,Allowances,float64,0
5,Total_Compensation,float64,0
6,Currency,str,0


... (2 outputs left out)


PERFORMANCE — 26,305 rows


,Column,Data_Type,Missing_Values
0,Employee_ID,str,0
1,Review_Year,int64,0
2,Performance_Rating,float64,0
3,Performance_Category,str,0



PROMOTIONS — 2,605 rows


,Column,Data_Type,Missing_Values
0,Employee_ID,str,0
1,Promotion_Date,datetime64[us],0
2,Previous_Level,int64,0
3,New_Level,int64,0
4,Salary_Increase_Pct,float64,0


In [21]:

%pip install sqlalchemy pymysql


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:

import sqlalchemy
import pymysql

print("SQLAlchemy Version:", sqlalchemy.__version__)
print("PyMySQL Version:", pymysql.__version__)


SQLAlchemy Version: 2.1.4
PyMySQL Version: 2.2.8


In [23]:

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from getpass import getpass

# MySQL connection details
mysql_user = "root"
mysql_host = "localhost"
mysql_port = 3306
database_name = "Compensation_Pay_Equity_DB"

# Enter password securely
mysql_password = getpass("Enter MySQL Password: ")

# Create connection URL
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=mysql_user,
    password=mysql_password,
    host=mysql_host,
    port=mysql_port,
    database=database_name
)

# Create database engine
engine = create_engine(connection_url)

# Test connection
try:
    with engine.connect() as connection:
        result = connection.execute(
            text("SELECT DATABASE()")
        )

        print("MySQL Connection Successful!")
        print("Connected Database:", result.scalar())

except Exception as e:
    print("Connection Failed!")
    print("Error:", e)


MySQL Connection Successful!
Connected Database: compensation_pay_equity_db


In [24]:

# Load cleaned Jobs DataFrame into MySQL

jobs.to_sql(
    name="jobs",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=1000
)

print("Jobs data loaded successfully!")

# Verify imported row count
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM jobs")
    )

    print("MySQL Jobs Records:", result.scalar())
    print("Python Jobs Records:", len(jobs))


Jobs data loaded successfully!
MySQL Jobs Records: 50
Python Jobs Records: 50


In [25]:

# Verify the Employees table is empty before loading
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM employees")
    ).scalar()

print("Existing MySQL Employee Records:", existing_rows)

if existing_rows == 0:

    # Load cleaned Employees DataFrame
    employees.to_sql(
        name="employees",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print("Employees data loaded successfully!")

else:
    print("Employees table already contains data. Load skipped.")

# Verify imported records
with engine.connect() as connection:
    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM employees")
    ).scalar()

    missing_experience = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM employees
            WHERE Years_Experience IS NULL
        """)
    ).scalar()

print("\nMySQL Employee Records:", mysql_count)
print("Python Employee Records:", len(employees))
print("Missing Experience in MySQL:", missing_experience)


Existing MySQL Employee Records: 0
Employees data loaded successfully!

MySQL Employee Records: 10000
Python Employee Records: 10000
Missing Experience in MySQL: 25


In [26]:

# Check existing Salary Bands records
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM salary_bands")
    ).scalar()

print("Existing MySQL Salary Band Records:", existing_rows)

# Load data only if the table is empty
if existing_rows == 0:

    salary_bands.to_sql(
        name="salary_bands",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print("Salary Bands data loaded successfully!")

else:
    print("Salary Bands table already contains data. Load skipped.")

# Verify imported records
with engine.connect() as connection:
    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM salary_bands")
    ).scalar()

    invalid_bands = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM salary_bands
            WHERE Band_Min > Band_Mid
               OR Band_Mid > Band_Max
        """)
    ).scalar()

print("\nMySQL Salary Band Records:", mysql_count)
print("Python Salary Band Records:", len(salary_bands))
print("Invalid Salary Band Structures:", invalid_bands)


Existing MySQL Salary Band Records: 0
Salary Bands data loaded successfully!

MySQL Salary Band Records: 300
Python Salary Band Records: 300
Invalid Salary Band Structures: 0


In [27]:

# Check existing Compensation records
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM compensation")
    ).scalar()

print("Existing MySQL Compensation Records:", existing_rows)

# Load data only if the table is empty
if existing_rows == 0:

    compensation.to_sql(
        name="compensation",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print("Compensation data loaded successfully!")

else:
    print("Compensation table already contains data. Load skipped.")

# Verify imported records
with engine.connect() as connection:

    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM compensation")
    ).scalar()

    missing_values = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM compensation
            WHERE Base_Salary IS NULL
               OR Annual_Bonus IS NULL
               OR Allowances IS NULL
               OR Total_Compensation IS NULL
        """)
    ).scalar()

    reconciliation_issues = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM compensation
            WHERE ABS(
                Total_Compensation
                - (Base_Salary + Annual_Bonus + Allowances)
            ) > 0.01
        """)
    ).scalar()

print("\nMySQL Compensation Records:", mysql_count)
print("Python Compensation Records:", len(compensation))
print("Missing Compensation Values:", missing_values)
print("Reconciliation Differences Above ₹0.01:", reconciliation_issues)


Existing MySQL Compensation Records: 0
Compensation data loaded successfully!

MySQL Compensation Records: 26305
Python Compensation Records: 26305
Missing Compensation Values: 0
Reconciliation Differences Above ₹0.01: 0


In [28]:

# Check existing Performance records
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM performance")
    ).scalar()

print("Existing MySQL Performance Records:", existing_rows)

# Load data only if the table is empty
if existing_rows == 0:

    performance.to_sql(
        name="performance",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print("Performance data loaded successfully!")

else:
    print("Performance table already contains data. Load skipped.")

# Verify imported records
with engine.connect() as connection:

    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM performance")
    ).scalar()

    missing_ratings = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM performance
            WHERE Performance_Rating IS NULL
        """)
    ).scalar()

    invalid_ratings = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM performance
            WHERE Performance_Rating NOT BETWEEN 1 AND 5
        """)
    ).scalar()

    unmatched_records = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM performance p
            LEFT JOIN compensation c
                ON p.Employee_ID = c.Employee_ID
               AND p.Review_Year = c.Compensation_Year
            WHERE c.Employee_ID IS NULL
        """)
    ).scalar()

print("\nMySQL Performance Records:", mysql_count)
print("Python Performance Records:", len(performance))
print("Missing Performance Ratings:", missing_ratings)
print("Invalid Performance Ratings:", invalid_ratings)
print("Unmatched Employee-Year Records:", unmatched_records)


Existing MySQL Performance Records: 0
Performance data loaded successfully!

MySQL Performance Records: 26305
Python Performance Records: 26305
Missing Performance Ratings: 0
Invalid Performance Ratings: 0
Unmatched Employee-Year Records: 0


In [29]:

# Check existing Promotions records
with engine.connect() as connection:
    existing_rows = connection.execute(
        text("SELECT COUNT(*) FROM promotions")
    ).scalar()

print("Existing MySQL Promotion Records:", existing_rows)

# Load data only if the table is empty
if existing_rows == 0:

    promotions.to_sql(
        name="promotions",
        con=engine,
        if_exists="append",
        index=False,
        chunksize=1000
    )

    print("Promotions data loaded successfully!")

else:
    print("Promotions table already contains data. Load skipped.")

# Verify imported records
with engine.connect() as connection:

    mysql_count = connection.execute(
        text("SELECT COUNT(*) FROM promotions")
    ).scalar()

    missing_values = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM promotions
            WHERE Employee_ID IS NULL
               OR Promotion_Date IS NULL
               OR Previous_Level IS NULL
               OR New_Level IS NULL
               OR Salary_Increase_Pct IS NULL
        """)
    ).scalar()

    invalid_promotions = connection.execute(
        text("""
            SELECT COUNT(*)
            FROM promotions
            WHERE New_Level <= Previous_Level
               OR Salary_Increase_Pct <= 0
        """)
    ).scalar()

print("\nMySQL Promotion Records:", mysql_count)
print("Python Promotion Records:", len(promotions))
print("Missing Promotion Values:", missing_values)
print("Invalid Promotion Records:", invalid_promotions)


Existing MySQL Promotion Records: 0
Promotions data loaded successfully!

MySQL Promotion Records: 2605
Python Promotion Records: 2605
Missing Promotion Values: 0
Invalid Promotion Records: 0


## Final Data Cleaning Summary

### Objective

Prepare reliable, analysis-ready employee and compensation datasets for SQL Server and subsequent pay equity analysis.

### Cleaning Actions Completed

**1. Duplicate Removal**
- Removed 18 exact duplicate compensation records.
- Confirmed zero duplicate primary or composite keys.

**2. Missing-Value Treatment**
- Replaced 45 missing education levels with "Unknown".
- Recovered 75 missing annual bonuses using compensation components.
- Recovered 60 missing performance ratings using validated category mappings.
- Retained 25 missing professional experience values to avoid unsupported imputation.

**3. Historical Record Cleaning**
- Removed 3,695 compensation records dated before employee hire years.
- Removed 3,695 performance records dated before employee hire years.
- Removed 132 promotion records dated before employee hire dates.
- Preserved joining-year compensation and performance records.

**4. Experience Quality**
- Identified 2,332 employees whose organizational tenure exceeds reported professional experience.
- Created Experience_Quality_Flag to distinguish Valid, Review, and Missing records.
- Preserved original experience values for transparency.

**5. Financial Validation**
- Confirmed no missing or negative compensation components.
- Verified that compensation totals reconcile within ₹0.01.
- Preserved legitimate senior-level compensation values.

### Final Dataset Summary

| Dataset | Final Records |
|---|---:|
| Employees | 10,000 |
| Jobs | 50 |
| Compensation | 26,305 |
| Salary Bands | 300 |
| Performance | 26,305 |
| Promotions | 2,605 |

### Remaining Limitations

- 25 professional experience values remain missing.
- 2,332 experience values require review before experience-adjusted pay comparisons.
- Historical job assignments are unavailable, limiting retrospective salary-band comparisons.
- Employee coverage differs across reporting years because pre-hire records were excluded.
- Annual records from an employee's joining year were retained without verifying their exact payroll or review dates.

### Final Conclusion

The six datasets have passed the defined cleaning and structural validation checks and are ready for the next stage of the analytics workflow, subject to the documented limitations.

All transformations were performed in memory using Python and Pandas. Original source CSV files remain unchanged, and no processed CSV files were created.